# DATA SCIENCE METHODOLOGY – 10 BƯỚC
## Phân tích kết quả học tập của sinh viên bằng SQL và Python
**Bộ dữ liệu:** `student_performance_dataset.csv` (1.000 bản ghi, 12 cột). **Trọng tâm:** truy xuất, làm sạch, phân tích dữ liệu SQL/Python. 10 bước tương ứng sơ đồ FPT University.

**Cách chạy:** đặt notebook cùng thư mục CSV, chọn **Run All**. Chỉ dùng thư viện `pandas`, `numpy` và `sqlite3` (sqlite3 có sẵn trong Python).

## Bước 1 — Business Understanding: Đặt mục tiêu
**Vấn đề:** Những đặc điểm học tập nào đi cùng với điểm thi cuối kỳ cao hoặc thấp?

**Câu hỏi phân tích:** (1) Điểm thi trung bình của sinh viên là bao nhiêu? (2) Các nhóm thời gian học và chuyên cần khác nhau có điểm khác nhau như thế nào? (3) Kết quả khác nhau ra sao giữa các nhóm học vấn phụ huynh và ngoại khóa?

**Đầu ra:** dữ liệu đã làm sạch, truy vấn SQL, bảng thống kê và báo cáo tóm tắt. Chỉ phân tích mối liên hệ; không khẳng định quan hệ nhân quả.

## Bước 2 — Analytic Approach: Hướng tiếp cận
Sử dụng **SQL** để kiểm tra dữ liệu, lọc, JOIN, GROUP BY, CASE WHEN, CTE và window functions. Sử dụng **Python/Pandas** để làm sạch, đối chiếu kết quả SQL và phân tích định lượng đơn giản. Không cần mô hình học máy.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("student_performance_dataset.csv")
assert DATA.exists(), f"Không tìm thấy {DATA}. Đặt CSV cùng thư mục notebook."
conn = sqlite3.connect("student_analysis.db")
print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.46.1


## Bước 3 — Data Requirements: Xác định dữ liệu cần dùng
Nhóm biến đầu vào: `study_time_hours`, `attendance_percent`, `sleep_hours`, `previous_grade`, `parental_education`, `extracurricular_activities`; biến kết quả: `final_exam_score`, `final_grade`. Khóa bản ghi: `student_id`. Các trường khác được giữ để hỗ trợ truy vấn.

In [2]:
raw = pd.read_csv(DATA)
print("Kích thước:", raw.shape)
print("Các cột:", raw.columns.tolist())
print(raw.head(5).to_string(index=False))
raw.to_sql("raw_students", conn, if_exists="replace", index=False)
print("Đã nạp dữ liệu vào SQLite: raw_students")

Kích thước: (1000, 12)
Các cột: ['student_id', 'gender', 'study_time_hours', 'attendance_percent', 'sleep_hours', 'parental_education', 'internet_access', 'extracurricular_activities', 'part_time_job', 'previous_grade', 'final_exam_score', 'final_grade']
 student_id gender  study_time_hours  attendance_percent  sleep_hours parental_education internet_access extracurricular_activities part_time_job  previous_grade  final_exam_score final_grade
          1   Male               4.0                98.0          6.5          Bachelors             Yes                        Yes            No            76.9             100.0           A
          2 Female               6.3               100.0          5.7        High School             Yes                        Yes           Yes            75.5             100.0           A
          3   Male               4.9                85.3          7.9          Bachelors             Yes                         No           Yes            88.5        

## Bước 4 — Data Collection: Truy xuất dữ liệu bằng SQL và Python
`pd.read_csv` đọc CSV; `to_sql` nạp vào SQLite; `pd.read_sql_query` lấy kết quả truy vấn từ SQLite ra DataFrame Python.

In [3]:
sql_4 = """
SELECT student_id, gender, study_time_hours, attendance_percent,
       parental_education, final_exam_score, final_grade
FROM raw_students
WHERE attendance_percent >= 90
ORDER BY final_exam_score DESC, student_id
LIMIT 10;
"""
print(pd.read_sql_query(sql_4,conn).to_string(index=False))

 student_id gender  study_time_hours  attendance_percent parental_education  final_exam_score final_grade
          1   Male               4.0                98.0          Bachelors             100.0           A
          2 Female               6.3               100.0        High School             100.0           A
         22   Male               3.3               100.0               None             100.0           A
         33 Female               5.3               100.0            Masters             100.0           A
         35 Female               6.6                90.4            Masters             100.0           A
        109 Female               5.8               100.0        High School             100.0           A
        131 Female               4.3               100.0        High School             100.0           A
        192   Male               4.7                96.6          Bachelors             100.0           A
        234 Female               3.8          

## Bước 5 — Data Understanding: Kiểm tra chất lượng dữ liệu bằng SQL & Python
Kiểm tra số dòng, ID trùng, giá trị NULL, phạm vi giá trị hợp lý và số lượng từng nhóm.

In [4]:
checks = {
 "Tổng số dòng": "SELECT COUNT(*) AS n FROM raw_students",
 "ID trùng": "SELECT student_id, COUNT(*) n FROM raw_students GROUP BY student_id HAVING COUNT(*) > 1",
 "Thiếu học vấn phụ huynh": "SELECT COUNT(*) n FROM raw_students WHERE parental_education IS NULL OR TRIM(parental_education) = ''",
 "Điểm ngoài 0-100": "SELECT COUNT(*) n FROM raw_students WHERE final_exam_score NOT BETWEEN 0 AND 100",
 "Chuyên cần ngoài 0-100": "SELECT COUNT(*) n FROM raw_students WHERE attendance_percent NOT BETWEEN 0 AND 100"
}
for label, query in checks.items():
    print("\n",label, "\n",pd.read_sql_query(query,conn).to_string(index=False))
print("\nNULL theo cột (Python):\n",raw.isna().sum().to_string())
print("Bản ghi trùng toàn bộ:",raw.duplicated().sum())


 Tổng số dòng 
    n
1000

 ID trùng 
 Empty DataFrame
Columns: [student_id, n]
Index: []

 Thiếu học vấn phụ huynh 
   n
102

 Điểm ngoài 0-100 
  n
 0

 Chuyên cần ngoài 0-100 
  n
 0

NULL theo cột (Python):
 student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
Bản ghi trùng toàn bộ: 0


## Bước 6 — Data Preparation: Làm sạch bằng SQL + Python (PHẦN TRỌNG TÂM)
**Nguyên tắc:** Không tự ý thay đổi điểm thi; đánh dấu điểm không hợp lệ thành NULL để kiểm tra; chuẩn hóa văn bản; điền `Unknown` cho học vấn phụ huynh thiếu; loại ID trùng nếu có (giữ bản ghi đầu). Câu lệnh SQL dưới đây thực hiện làm sạch trên **bảng mới**, không sửa dữ liệu gốc.

In [5]:
conn.executescript("""
DROP TABLE IF EXISTS students_clean_sql;
CREATE TABLE students_clean_sql AS
WITH standardized AS (
 SELECT student_id,
        TRIM(gender) AS gender,
        CASE WHEN study_time_hours BETWEEN 0 AND 24 THEN study_time_hours ELSE NULL END AS study_time_hours,
        CASE WHEN attendance_percent BETWEEN 0 AND 100 THEN attendance_percent ELSE NULL END AS attendance_percent,
        CASE WHEN sleep_hours BETWEEN 0 AND 24 THEN sleep_hours ELSE NULL END AS sleep_hours,
        COALESCE(NULLIF(TRIM(parental_education), ''), 'Unknown') AS parental_education,
        TRIM(internet_access) AS internet_access,
        TRIM(extracurricular_activities) AS extracurricular_activities,
        TRIM(part_time_job) AS part_time_job,
        CASE WHEN previous_grade BETWEEN 0 AND 100 THEN previous_grade ELSE NULL END AS previous_grade,
        CASE WHEN final_exam_score BETWEEN 0 AND 100 THEN final_exam_score ELSE NULL END AS final_exam_score,
        TRIM(final_grade) AS final_grade,
        ROW_NUMBER() OVER (PARTITION BY student_id ORDER BY rowid) AS rn
 FROM raw_students
)
SELECT student_id, gender, study_time_hours, attendance_percent, sleep_hours,
       parental_education, internet_access, extracurricular_activities,
       part_time_job, previous_grade, final_exam_score, final_grade
FROM standardized WHERE rn = 1 AND student_id IS NOT NULL;
""")
clean_sql = pd.read_sql_query("SELECT * FROM students_clean_sql ORDER BY student_id",conn)
print("SQL - số dòng sạch:", len(clean_sql))
print("SQL - học vấn Unknown:", (clean_sql.parental_education == "Unknown").sum())

SQL - số dòng sạch: 1000
SQL - học vấn Unknown: 102


In [6]:
clean_py = raw.copy()
clean_py.columns = clean_py.columns.str.strip().str.lower()
clean_py = clean_py.dropna(subset=["student_id"]).drop_duplicates(subset=["student_id"], keep="first")
for column in clean_py.select_dtypes(include="object").columns:
    clean_py[column] = clean_py[column].astype("string").str.strip()
    clean_py[column] = clean_py[column].replace("", pd.NA)
clean_py["parental_education"] = clean_py["parental_education"].fillna("Unknown")
for column,low,high in [("study_time_hours",0,24),("sleep_hours",0,24),
                        ("attendance_percent",0,100),("previous_grade",0,100),
                        ("final_exam_score",0,100)]:
    clean_py[column] = pd.to_numeric(clean_py[column],errors="coerce")
    clean_py.loc[~clean_py[column].between(low,high), column] = np.nan
clean_py = clean_py.sort_values("student_id").reset_index(drop=True)
print("Python - số dòng sạch:", len(clean_py))
print("Python - số ô thiếu còn lại:\n",clean_py.isna().sum().to_string())
print("SQL vs Python - học vấn Unknown:",(clean_sql.parental_education == "Unknown").sum(),(clean_py.parental_education == "Unknown").sum())
clean_py.to_sql("students_clean_python",conn,if_exists="replace",index=False)

Python - số dòng sạch: 1000
Python - số ô thiếu còn lại:
 student_id                    0
gender                        0
study_time_hours              0
attendance_percent            0
sleep_hours                   0
parental_education            0
internet_access               0
extracurricular_activities    0
part_time_job                 0
previous_grade                0
final_exam_score              0
final_grade                   0
SQL vs Python - học vấn Unknown: 102 102


1000

## Bước 7 — Modeling: Phân nhóm quy tắc bằng SQL và Python
Để ưu tiên SQL/Python, bước “Modeling” sử dụng **mô hình phân nhóm quy tắc** theo thời gian học (`<2h`, `2–4h`, `>4h`), không cần học máy.

In [7]:
sql_7 = """
WITH grouped AS (
 SELECT CASE WHEN study_time_hours < 2 THEN 'Dưới 2h'
             WHEN study_time_hours <= 4 THEN '2 đến 4h'
             ELSE 'Trên 4h' END AS study_group,
        final_exam_score, attendance_percent
 FROM students_clean_sql
 WHERE study_time_hours IS NOT NULL AND final_exam_score IS NOT NULL
)
SELECT study_group, COUNT(*) AS student_count,
       ROUND(AVG(final_exam_score),2) AS avg_exam_score,
       ROUND(AVG(attendance_percent),2) AS avg_attendance
FROM grouped GROUP BY study_group
ORDER BY CASE study_group WHEN 'Dưới 2h' THEN 1 WHEN '2 đến 4h' THEN 2 ELSE 3 END;
"""
result_study = pd.read_sql_query(sql_7,conn)
print("SQL - Điểm theo nhóm thời gian học:\n",result_study.to_string(index=False))
clean_py["study_group"] = np.select([clean_py.study_time_hours < 2, clean_py.study_time_hours <= 4],
    ["Dưới 2h", "2 đến 4h"], default="Trên 4h")
print("\nPython - Tương quan Pearson:\n",clean_py[["study_time_hours","attendance_percent","sleep_hours","previous_grade","final_exam_score"]].corr()["final_exam_score"].round(3).to_string())

SQL - Điểm theo nhóm thời gian học:
 study_group  student_count  avg_exam_score  avg_attendance
    Dưới 2h            144           74.79           86.17
   2 đến 4h            489           81.36           85.26
    Trên 4h            367           89.88           84.45

Python - Tương quan Pearson:
 study_time_hours      0.568
attendance_percent    0.262
sleep_hours           0.148
previous_grade        0.406
final_exam_score      1.000


## Bước 8 — Evaluation: Đánh giá và đối chiếu bằng SQL/Python
Đối chiếu số dòng và điểm trung bình SQL/Python; đánh giá dữ liệu còn thiếu, không kết luận yếu tố nào “gây ra” điểm số.

In [8]:
sql_8 = """
SELECT COUNT(*) AS total_students, ROUND(AVG(final_exam_score),3) AS avg_score,
       SUM(CASE WHEN parental_education = 'Unknown' THEN 1 ELSE 0 END) AS unknown_education,
       SUM(CASE WHEN final_exam_score IS NULL THEN 1 ELSE 0 END) AS missing_exam_score
FROM students_clean_sql;
"""
summary_sql = pd.read_sql_query(sql_8,conn)
print("SQL kiểm tra:\n",summary_sql.to_string(index=False))
print("Python điểm trung bình:",round(clean_py.final_exam_score.mean(),3))
assert len(clean_sql) == len(clean_py), "SQL và Python không đồng nhất số bản ghi"
assert abs(clean_sql.final_exam_score.mean()-clean_py.final_exam_score.mean()) < 1e-8
print("Đối chiếu SQL/Python: ĐẠT")
print("\nSQL - Điểm theo tỷ lệ chuyên cần:\n",pd.read_sql_query("""
SELECT CASE WHEN attendance_percent < 80 THEN 'Dưới 80%'
            WHEN attendance_percent < 90 THEN '80-<90%' ELSE '90-100%' END AS attendance_group,
       COUNT(*) AS students, ROUND(AVG(final_exam_score),2) AS avg_score
FROM students_clean_sql GROUP BY attendance_group ORDER BY avg_score DESC
""",conn).to_string(index=False))

SQL kiểm tra:
  total_students  avg_score  unknown_education  missing_exam_score
           1000     83.543                102                   0
Python điểm trung bình: 83.544
Đối chiếu SQL/Python: ĐẠT

SQL - Điểm theo tỷ lệ chuyên cần:
 attendance_group  students  avg_score
         90-100%       326      86.41
         80-<90%       389      83.40
        Dưới 80%       285      80.47


## Bước 9 — Deployment: Xuất kết quả để báo cáo
Lưu dữ liệu sạch và bảng phân tích ra CSV; database SQLite đã chứa `raw_students`, `students_clean_sql`, `students_clean_python`.

In [9]:
clean_py.to_csv("student_performance_cleaned_python.csv",index=False,encoding="utf-8-sig")
clean_sql.to_csv("student_performance_cleaned_sql.csv",index=False,encoding="utf-8-sig")
result_study.to_csv("analysis_study_groups.csv",index=False,encoding="utf-8-sig")
conn.commit()
print("Xuất 3 file CSV và student_analysis.db thành công")

Xuất 3 file CSV và student_analysis.db thành công


## Bước 10 — Feedback: Kết luận, kiểm tra lại và cải tiến
- Tiếp tục kiểm tra dữ liệu thiếu/ngoại lệ khi nạp phiên bản mới.
- Có thể bổ sung các truy vấn nhóm theo giới tính, học vấn phụ huynh, hoạt động ngoại khóa.
- **Lưu ý quan trọng:** `student_performance_updated_1000.csv` là bộ dữ liệu khác; không mặc định ghép theo vị trí dòng nếu chưa xác nhận khóa chung và ý nghĩa biến.
- Sản phẩm đã hoàn thành quy trình 10 bước; trọng tâm nằm ở SQL truy xuất/làm sạch và Python xử lý/phân tích.

In [10]:
print("KẾT QUẢ CUỐI CÙNG")
print("Số bản ghi:",len(clean_py))
print("Điểm cuối kỳ trung bình:",round(clean_py.final_exam_score.mean(),2))
print("Giờ học trung bình:",round(clean_py.study_time_hours.mean(),2))
print("Điểm theo nhóm học:\n",result_study.to_string(index=False))
conn.close()

KẾT QUẢ CUỐI CÙNG
Số bản ghi: 1000
Điểm cuối kỳ trung bình: 83.54
Giờ học trung bình: 3.57
Điểm theo nhóm học:
 study_group  student_count  avg_exam_score  avg_attendance
    Dưới 2h            144           74.79           86.17
   2 đến 4h            489           81.36           85.26
    Trên 4h            367           89.88           84.45
